<a href="https://colab.research.google.com/github/rohanmyers/queue-viability/blob/main/notebooks/04_baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = Path("/content/drive/MyDrive/queue-viability")
RAW = DRIVE_ROOT / "data" / "raw"
INTERIM = DRIVE_ROOT / "data" / "interim"
PROCESSED = DRIVE_ROOT / "data" / "processed"
FIGURES = DRIVE_ROOT / "reports" / "figures"

Mounted at /content/drive


In [56]:
!pip install -q pyarrow openpyxl

import pandas as pd
import numpy as np
from pathlib import Path

features = pd.read_parquet(INTERIM / "features_v1.parquet")
print(f"Loaded {len(features):,} rows × {features.shape[1]} cols")

Loaded 33,566 rows × 23 cols


In [57]:
EIA_TABLE_4 = RAW / "table_4.xlsx"

# This file has 2 header rows (title + subtitle) before the actual headers
state_price_raw = pd.read_excel(EIA_TABLE_4, sheet_name="Table 4", header=2)
print(state_price_raw.columns.tolist())
print(state_price_raw.head())

['State', 'Residential', 'Commercial', 'Industrial', 'Transportation', 'Total']
           State  Residential  Commercial  Industrial Transportation  \
0    New England    27.678886   20.554466   16.282883      12.992054   
1    Connecticut    28.745463   21.209726   17.122955      17.767971   
2          Maine    24.286709   18.217494   12.464281              .   
3  Massachusetts    29.349701   20.895059   18.190619       9.215189   
4  New Hampshire    23.404467   19.398687   16.210857              .   

       Total  
0  23.029285  
1  24.369524  
2  19.662481  
3  23.937668  
4  20.604971  


In [58]:
# Filter out census-region aggregate rows (e.g., "New England", "Middle Atlantic")
# These are sums across states — keep only the actual state rows
state_aggregates = [
    'New England', 'Middle Atlantic', 'East North Central', 'West North Central',
    'South Atlantic', 'East South Central', 'West South Central',
    'Mountain', 'Pacific Contiguous', 'Pacific Noncontiguous', 'U.S. Total'
]

state_price = state_price_raw[~state_price_raw['State'].isin(state_aggregates)].copy()
state_price = state_price[['State', 'Industrial']].rename(
    columns={'Industrial': 'industrial_price_cents_kwh'}
)

# Convert the District of Columbia to its 2-letter abbreviation
state_price['State'] = state_price['State'].replace({'District of Columbia': 'DC'})

# The state names in the EIA file are full names (e.g., "California"),
# but your LBNL data uses 2-letter codes. Map them.
us_state_to_abbrev = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI",
    "South Carolina": "SC", "South Dakota": "SD", "Tennessee": "TN",
    "Texas": "TX", "Utah": "UT", "Vermont": "VT", "Virginia": "VA",
    "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI",
    "Wyoming": "WY", "DC": "DC",
}

state_price['State'] = state_price['State'].map(us_state_to_abbrev)
state_price = state_price.dropna(subset=['State'])  # drop unmapped rows

# Convert price column to numeric (the file has some "." values for Transportation;
# Industrial should be all numeric, but be safe)
state_price['industrial_price_cents_kwh'] = pd.to_numeric(
    state_price['industrial_price_cents_kwh'], errors='coerce'
)

print(f"Final state price table: {len(state_price)} rows")
print(state_price.sort_values('industrial_price_cents_kwh').to_string())

Final state price table: 51 rows
   State  industrial_price_cents_kwh
51    NM                    5.430662
42    LA                    5.614528
43    OK                    5.841376
44    TX                    6.121500
39    TN                    6.209165
37    KY                    6.498166
57    WA                    6.609028
41    AR                    6.610511
18    IA                    6.804213
38    MS                    6.813471
32    SC                    6.842986
15    OH                    7.096382
29    GA                    7.211297
23    ND                    7.246613
36    AL                    7.253566
49    MT                    7.585923
22    NE                    7.662630
48    ID                    7.693207
19    KS                    7.731173
31    NC                    7.767010
34    WV                    7.809345
52    UT                    7.855137
10    PA                    7.865209
21    MO                    7.870943
46    AZ                    7.901039
53   

In [59]:
features['state'] = features['state'].astype(str).str.upper().str.strip()

features = features.merge(
    state_price, left_on='state', right_on='State', how='left'
).drop(columns=['State'])

print(f"Rows with price: {features['industrial_price_cents_kwh'].notna().sum():,} / {len(features):,}")
print(features['industrial_price_cents_kwh'].describe())

Rows with price: 33,485 / 33,566
count    33485.000000
mean         9.184754
std          4.090543
min          5.430662
25%          7.096382
50%          7.901039
75%          8.994044
max         21.527795
Name: industrial_price_cents_kwh, dtype: float64


In [60]:
missing_price_states = features[features['industrial_price_cents_kwh'].isna()]['state'].unique()
print(missing_price_states)

['NONE' 'MX']


In [61]:
features = features[~features['state'].isin(['NONE', 'MX'])].copy()
print(f"Shape after removing 'NONE' and 'MX' states: {features.shape}")

Shape after removing 'NONE' and 'MX' states: (33485, 24)


In [63]:
features = features[features['q_year'].fillna(0) >= 2000]
print(f"Shape after removing records before 2000: {features.shape}")

Shape after removing records before 2000: (32536, 24)


In [64]:
# Capacity-class feature (small/medium/large/huge)
features['capacity_class'] = pd.cut(
    features['mw_total'],
    bins=[0, 50, 200, 500, np.inf],
    labels=['small', 'medium', 'large', 'huge']
).astype(str)

# Region × fuel interaction as a categorical
features['region_fuel'] = features['region'] + '_' + features['fuel_bucketed']

# Has the developer-history feature actually populated?
features['has_dev_history'] = features['dev_prior_n'].fillna(0).gt(0).astype(int)

print(features[['capacity_class', 'has_dev_history']].head())

    capacity_class  has_dev_history
365          small                0
366          small                0
367           huge                1
368          small                0
369           huge                0


In [65]:
features['split'] = np.where(
    ~features['is_trainable'], 'exclude',
    np.where(features['q_year'] <= 2017, 'train', 'test')
)

print(features['split'].value_counts())
print(f"\nTrain: {(features['split']=='train').sum():,} rows")
print(f"Test:  {(features['split']=='test').sum():,} rows")
print(f"To score: {features['is_scoring'].sum():,} rows")

# Verify class balance in each
for split in ['train', 'test']:
    sub = features[features['split'] == split]
    pos_rate = sub['y'].mean()
    print(f"{split} positive rate: {pos_rate:.1%}  (n={len(sub):,})")

split
exclude    17258
train      12100
test        3178
Name: count, dtype: int64

Train: 12,100 rows
Test:  3,178 rows
To score: 10,778 rows
train positive rate: 21.7%  (n=12,100)
test positive rate: 13.4%  (n=3,178)


In [47]:
final_cols = [
    'q_id', 'split', 'y', 'is_scoring',
    # Direct features
    'fuel_bucketed', 'region', 'state', 'service', 'project_type',
    'capacity_class',
    # Numeric features
    'mw_total', 'log_mw', 'q_year', 'proposed_dev_years',
    # Engineered features
    'dev_prior_n', 'dev_prior_completion_rate', 'has_dev_history',
    'cluster_size', 'q_year_region_volume',
    # External features
    'industrial_price_cents_kwh',
    # Interaction
    'region_fuel',
]

final = features[final_cols].copy()

# Cast object columns to string for parquet safety
for col in final.select_dtypes(include='object').columns:
    final[col] = final[col].astype(str)

# Save full + splits
final.to_parquet(PROCESSED / "features_final.parquet", index=False)
final[final['split'] == 'train'].to_parquet(PROCESSED / "train.parquet", index=False)
final[final['split'] == 'test'].to_parquet(PROCESSED / "test.parquet", index=False)
final[final['is_scoring']].to_parquet(PROCESSED / "to_score.parquet", index=False)

# Verify saves
import time; time.sleep(3)
for f in ['features_final', 'train', 'test', 'to_score']:
    df = pd.read_parquet(PROCESSED / f"{f}.parquet")
    print(f"{f:18s}: {df.shape}")

features_final    : (33485, 21)
train             : (12456, 21)
test              : (3178, 21)
to_score          : (10943, 21)


In [48]:
train = pd.read_parquet(PROCESSED / "train.parquet")

print("=== NUMERIC FEATURES ===")
numeric_features = [
    'log_mw', 'q_year', 'proposed_dev_years',
    'dev_prior_n', 'dev_prior_completion_rate',
    'cluster_size', 'q_year_region_volume',
    'industrial_price_cents_kwh', 'has_dev_history',
]
for col in numeric_features:
    if col in train.columns:
        valid = train[[col, 'y']].dropna()
        if len(valid) > 100:
            corr = valid.corr().iloc[0, 1]
            print(f"  {col:35s}  corr={corr:+.3f}  (n={len(valid):,})")

print("\n=== CATEGORICAL FEATURES (completion rate by category) ===")
for col in ['fuel_bucketed', 'region', 'service', 'capacity_class']:
    print(f"\n{col}:")
    rates = train.groupby(col)['y'].agg(['mean', 'count']).sort_values('mean')
    print(rates)

=== NUMERIC FEATURES ===
  log_mw                               corr=-0.126  (n=12,454)
  q_year                               corr=-0.147  (n=12,456)
  proposed_dev_years                   corr=+nan  (n=8,876)
  dev_prior_n                          corr=+0.048  (n=12,456)
  dev_prior_completion_rate            corr=+0.472  (n=311)
  cluster_size                         corr=-0.027  (n=12,456)
  q_year_region_volume                 corr=-0.086  (n=12,456)
  industrial_price_cents_kwh           corr=-0.035  (n=12,456)
  has_dev_history                      corr=+0.085  (n=12,456)

=== CATEGORICAL FEATURES (completion rate by category) ===

fuel_bucketed:
                    mean  count
fuel_bucketed                  
Battery         0.150470    319
Solar           0.175697   4485
Wind            0.205451   3339
Other           0.235415    977
Gas             0.302257   2038
Other_combined  0.314387    563
Coal            0.347985    273
Solar+Battery   0.352113    142
Hydro           0.

In [49]:
print(train['proposed_dev_years'].describe())
print(f"\nNaN count: {train['proposed_dev_years'].isna().sum():,} / {len(train):,}")
print(f"Sample values:")
print(train['proposed_dev_years'].dropna().head(20))
print(f"\nDtype: {train['proposed_dev_years'].dtype}")

count    8876.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: proposed_dev_years, dtype: float64

NaN count: 3,580 / 12,456
Sample values:
0     0.0
5     0.0
6     0.0
7     0.0
8     0.0
11    0.0
12    0.0
13    0.0
14    0.0
15    0.0
16    0.0
17    0.0
18    0.0
19    0.0
20    0.0
21    0.0
22    0.0
23    0.0
24    0.0
25    0.0
Name: proposed_dev_years, dtype: float64

Dtype: float64


In [50]:
train['proposed_dev_years'] = pd.to_numeric(train['proposed_dev_years'], errors='coerce')
corr = train[['proposed_dev_years', 'y']].dropna().corr().iloc[0, 1]
print(f"Fixed correlation: {corr:+.3f}")

Fixed correlation: +nan


In [51]:
print("ERCOT year distribution in train:")
print(train[train['region'] == 'ERCOT']['q_year'].value_counts().sort_index())

ERCOT year distribution in train:
q_year
1970.0    25
2007.0     1
2009.0     4
2010.0     3
2011.0     3
2012.0     1
2013.0     8
2014.0    10
2015.0    16
2016.0    31
2017.0    73
Name: count, dtype: int64


In [52]:
# Reload the unprocessed data and look at the source columns
gen = pd.read_parquet(INTERIM / "generation_queue.parquet")

print("q_date:")
print(f"  dtype: {gen['q_date'].dtype}")
print(f"  non-null: {gen['q_date'].notna().sum():,}")
print(f"  sample: {gen['q_date'].dropna().iloc[0]}")

print("\nprop_date:")
print(f"  dtype: {gen['prop_date'].dtype}")
print(f"  non-null: {gen['prop_date'].notna().sum():,}")
print(f"  sample: {gen['prop_date'].dropna().iloc[0]}")

q_date:
  dtype: float64
  non-null: 32,977
  sample: 43511.0

prop_date:
  dtype: float64
  non-null: 26,800
  sample: 39539.0


In [53]:
gen['q_date'] = pd.to_datetime(gen['q_date'], errors='coerce')
gen['prop_date'] = pd.to_datetime(gen['prop_date'], errors='coerce')

gen['proposed_dev_years'] = (
    (gen['prop_date'] - gen['q_date']).dt.days / 365.25
).clip(lower=0, upper=15)

print(gen['proposed_dev_years'].describe())

count    26687.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: proposed_dev_years, dtype: float64
